In [7]:
%run ./board.ipynb

### Class UI

Com o objetivo de disponibilizar o jogo via terminal, para que o usuário conseguisse jogar contra outro usuário, e até observar duas IA's jogarem entre si. A necessidade de criar outputs em terminais era evidente.

Assim foi criada a classe UI, uma classe com métodos estáticos pois a classe não guarda em si nenhum atributo ou informação, é apenas uma segregação de todas as chamadas de print para fins de organização do espaço de trabalho.

Em primeiro lugar, nossa UI utilizará das seguintes bibliotecas:
- os, para acessar informações do sistema.
- time, para fins de animações nas jogadas.

In [8]:
import os
import time

A classe UI, é criada com o seu método mais recorrente:
- *clear_screen*, que identifica o sistema operacional do computador em que o código está rodando e insere o comando de limpeza de output do terminal.


In [9]:
class UI:
    @staticmethod
    def clear_screen():
        os.system('cls' if os.name == 'nt' else 'clear')
    
    @staticmethod
    def wait_for_enter(message="\nPress Enter to return..."):
        input(message)


Para o menu do jogo, seleção de modos, seleção de IA, disponibilização de regras, créditos, foi pensando na criação dos seguintes métodos: *display_main_menu*, *display_play_menu*, *display_ai_menu*, *display_rules* and *display_credits*.

In [10]:
@staticmethod
def display_main_menu():
    UI.clear_screen()
    print("===================================")
    print("  Welcome to PopOut on terminal!   ")
    print("===================================")
    print(" 1 - Play")
    print(" 2 - Rules")
    print(" 3 - Credits")
    print(" 4 - Exit Game")
    print("===================================")


@staticmethod
def display_play_menu():
    UI.clear_screen()
    print("===================================")
    print("           SELECT MODE             ")
    print("===================================")
    print(" 1 - Human Vs Human")
    print(" 2 - Human vs AI")
    print(" 3 - AI vs AI")
    print(" 4 - Back")
    print("===================================")


@staticmethod
def display_ai_menu(player_label):
    print(f"\n===================================")
    print(f" Select Algorithm for {player_label} ")
    print(f"===================================")
    print(" 1 - MCTS Heuristic (With heuristics and optimizations)")
    print(" 2 - MCTS Vanilla (Standard)")
    print(" 3 - MCTS Multi-Expansion (N-Children)")
    print("===================================")


@staticmethod
def display_rules():
    UI.clear_screen()
    print("===============================================================")
    print("                      POPOUT - RULES                           ")
    print("===============================================================")
    print("1. Standard Connect 4 rules apply: get 4 pieces in a row")
    print("   (horizontal, vertical, or diagonal) to win.")
    print("2. DROP: On your turn, you can drop a piece into the top of")
    print("   any column that is not full.")
    print("3. POP OUT: Instead of dropping, you can choose to remove")
    print("   (pop) one of YOUR OWN pieces from the VERY BOTTOM of a")
    print("   column. The pieces above it will drop down one space.")
    print("4. SIMULTANEOUS WIN (Rule 1): If popping a piece creates a")
    print("   win for both players, the player who popped the piece wins!")
    print("5. FULL BOARD (Rule 2): If the board is completely full, the")
    print("   current player must either pop a piece or declare a draw.")
    print("6. THREEFOLD REPETITION (Rule 3): If the exact same board state")
    print("   occurs 3 times, either player can declare a draw.")
    print("===============================================================")

@staticmethod
def display_credits():
    UI.clear_screen()
    print("===============================================================")
    print("                        CREDITS                                ")
    print("===============================================================")
    print(" Game developed by: Aly, Rafael and Victor.")
    print(" Variant: PopOut (Official Rules)")
    print(" Course/Context: Artificial Intelligence & Data Science")
    print("===============================================================")

# Monkey Patching #
UI.display_main_menu = display_main_menu
UI.display_play_menu = display_play_menu
UI.display_ai_menu = display_ai_menu
UI.display_rules = display_rules
UI.display_credits = display_credits

#Example of usage:
#UI.display_credits()

Para visualizar o tabuleiro, utilizou-se do método *print_board*, ele acessa o objeto da classe Board criada, e lê o atributo grid da classe para desenha-lo no terminal. Permitindo a visualização em tempo real do jogo.

Já o método *render* utiliza do método de limpar o terminal para imprimir a mensagem indicativa da jogada atual e utilizar do método acima para imprimir o board, melhorando a experiência do jogador.

In [ ]:
@staticmethod
def print_board(board):
    print("\n  1   2   3   4   5   6   7")
    print("|---|---|---|---|---|---|---|")
    for row in board.grid:
        print("| " + " | ".join(row) + " |")
        print("|---|---|---|---|---|---|---|") 
    print()

@staticmethod
def render(board, message="Connect 4 - Pop Out Variant."):
    """Centraliza a atualização do ecrã: limpa, mostra mensagem e imprime o tabuleiro."""
    UI.clear_screen()
    print(message)
    UI.print_board(board)

# Monkey Patching #
UI.print_board = print_board
UI.render = render

#Example of usage:
#Game = Board()
#UI.render(Game)

Por fim, para os movimentos animados implementados no nosso jogo, temos dois métodos, *animate_drop* o qual utiliza uma combinação de cópias e limpezas do terminal, em um determinado tempo para que haja a impressão de uma peça caindo em sua posição, e não somente a atualização do estado no board.

O outro método é o *animate_pop*, o qual utiliza da mesma lógica mas com o movimento de remoção de peça, onde a peça inferior é substituída por um espaço vazio e posteriormente a descida da coluna toda.

In [12]:
@staticmethod
def animate_drop(board, col, final_row, piece):
    """Anima a peça a cair pela coluna usando uma cópia visual temporária."""
    temp_board = board.copy() 
    
    for r in range(final_row):
        temp_board.grid[r][col] = piece
        UI.render(temp_board, "Connect 4 - Dropping Piece...")
        time.sleep(0.1) 
        temp_board.grid[r][col] = ' '
        
@staticmethod
def animate_pop(board, col):
    """Anima a remoção da peça usando uma cópia visual temporária."""
    temp_board = board.copy() 
    temp_board.grid[temp_board.rows - 1][col] = ' '
    
    UI.render(temp_board, "Connect 4 - Popping Piece Out...")
    time.sleep(0.4) 
    
    for r in range(temp_board.rows - 1, 0, -1):
        if temp_board.grid[r - 1][col] != ' ':
            temp_board.grid[r][col] = temp_board.grid[r - 1][col]
            temp_board.grid[r - 1][col] = ' ' 
            
            UI.render(temp_board, "Connect 4 - Pieces Falling...")
            time.sleep(0.2)

# Monkey Patch #

UI.animate_drop = animate_drop
UI.animate_pop = animate_pop